# Stochastic Short Rate Models — Deep Dive

A guided tour of the `stochastic-rate-models` library: Vasicek, CIR and Hull-White short rate models, exact vs. discretised simulation, analytical bond pricing, calibration, and risk.

Run from the repository root so that the packages import correctly.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd

from models import VasicekModel, VasicekParams, CIRModel, CIRParams, HullWhiteModel, HullWhiteParams
np.set_printoptions(precision=4, suppress=True)

## 1. The Vasicek model

$$dr(t) = \kappa(\theta - r(t))\,dt + \sigma\,dW(t)$$

Gaussian, mean-reverting, with a closed-form transition density and an affine term structure.

In [ ]:
vas = VasicekModel(VasicekParams(kappa=0.5, theta=0.05, sigma=0.02))
mats = np.array([0.5, 1, 2, 3, 5, 7, 10, 20, 30])
print('Long-run yield:', round(vas.long_run_yield(), 5))
print('Half-life (yrs):', round(vas.half_life(), 3))
print('P(r<0) at 5y:', round(vas.negative_rate_probability(0.03, 5.0), 4))
pd.DataFrame({'maturity': mats, 'yield': vas.yield_curve(0.03, mats), 'forward': vas.forward_curve(0.03, mats)})

### Exact vs. Euler simulation
The Vasicek transition is exactly Gaussian, so exact simulation carries **zero discretisation bias**.

In [ ]:
exact = vas.simulate_exact(0.03, 5.0, 50_000, 60, seed=1)
euler = vas.simulate_euler(0.03, 5.0, 50_000, 60, seed=1)
analytic_mean = vas.conditional_mean(0.03, 5.0)
print('Analytic E[r(5)] :', round(analytic_mean, 5))
print('Exact   E[r(5)]  :', round(exact.terminal.mean(), 5))
print('Euler   E[r(5)]  :', round(euler.terminal.mean(), 5))
print('Elapsed (exact/euler):', round(exact.elapsed_seconds, 3), round(euler.elapsed_seconds, 3))

## 2. The CIR model and the Feller condition

$$dr(t) = \kappa(\theta - r(t))\,dt + \sigma\sqrt{r(t)}\,dW(t)$$

Square-root diffusion keeps rates non-negative; the Feller condition $2\kappa\theta > \sigma^2$ keeps them strictly positive.

In [ ]:
cir = CIRModel(CIRParams(kappa=0.5, theta=0.05, sigma=0.1))
holds, lhs, rhs = cir.feller_condition_check()
print(f'Feller holds: {holds}  (2*k*th={lhs:.4f} vs sigma^2={rhs:.4f})')
sim = cir.simulate_exact(0.03, 5.0, 50_000, 60, seed=2)
print('All paths >= 0:', bool((sim.paths >= 0).all()))
print('E[r(5)] exact vs analytic:', round(sim.terminal.mean(), 5), round(cir.conditional_mean(0.03, 5.0), 5))

## 3. Hull-White: an exact fit to the market curve

$$dr(t) = (\theta(t) - \kappa r(t))\,dt + \sigma\,dW(t)$$

$\theta(t)$ is derived from the market forward curve, so today's bonds are repriced exactly.

In [ ]:
mkt_mats = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])
mkt_ylds = np.array([0.020, 0.022, 0.025, 0.028, 0.030, 0.032, 0.034, 0.035, 0.037, 0.038])
hw = HullWhiteModel(HullWhiteParams(kappa=0.1, sigma=0.01)).set_initial_curve(mkt_mats, mkt_ylds)
fit = hw.yield_curve_fitted(mkt_mats)
print('Fit RMSE (bps):', round(np.sqrt(np.mean((fit - mkt_ylds)**2)) * 1e4, 4))
print('Cap (5y, 3%):  ', round(hw.cap_price(0.03, 5.0, 1_000_000, 4), 2))
print('Floor (5y, 3%):', round(hw.floor_price(0.03, 5.0, 1_000_000, 4), 2))
print('Payer swaption 1y into 5y @3.5%:', round(hw.swaption_price(0.035, 5.0, 1.0, 1_000_000), 2))

## 4. Bond pricing, duration and risk

In [ ]:
from pricing import CouponBondPricer
from risk import DV01Engine
from scenarios import ParallelShiftScenario

bond = {'type': 'coupon', 'coupon_rate': 0.04, 'maturity': 10, 'freq': 2, 'face': 100}
bp = CouponBondPricer().price(vas, 0.03, 0.04, 10, 2, 100)
print(f'Price={bp.price:.4f}  ModDur={bp.modified_duration:.4f}  Convexity={bp.convexity:.4f}  DV01={bp.dv01:.5f}')
print('DV01 engine:', round(DV01Engine().dv01(vas, 0.03, bond), 5))
ParallelShiftScenario().run(vas, 0.03, bond)

## 5. Calibration round-trip
Simulate from known parameters, then recover them by OLS and MLE.

In [ ]:
from calibration import OLSCalibrator, MLECalibrator
series = vas.simulate_exact(0.04, 20.0, 1, 5040, seed=7).paths[0]
dt = 20.0 / 5040
print('OLS:', OLSCalibrator().calibrate_vasicek(series, dt)['params'])
print('MLE:', MLECalibrator().calibrate_vasicek(series, dt)['params'])

---
This notebook only scratches the surface — see the Streamlit app (`streamlit run app.py`) for the full interactive experience across simulation, curves, pricing, calibration, derivatives, risk and scenarios.